In [ ]:
!pip install optimum[onnxruntime] transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.3.7
    Uninstalling huggingface_hub-1.3.7:
      Successfully uninstalled huggingface_hub-1.3.7
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/notification_model/final_model"
ONNX_OUTPUT_PATH = "/content/drive/MyDrive/notification_model/distilbert_onnx"


In [ ]:
import os
os.makedirs(ONNX_OUTPUT_PATH, exist_ok=True)


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    export=True
)

model.save_pretrained(ONNX_OUTPUT_PATH)

print("ONNX model exported successfully.")


`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


ONNX model exported successfully.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained(ONNX_OUTPUT_PATH)

print("Tokenizer saved successfully.")


Tokenizer saved successfully.


In [ ]:
import json
import os

# Because you trained on priority values
label_list = [1, 2, 3, 4]

label_map = {i: label for i, label in enumerate(label_list)}

with open(os.path.join(ONNX_OUTPUT_PATH, "label_map.json"), "w") as f:
    json.dump(label_map, f)

print("Label map saved successfully.")


Label map saved successfully.


In [ ]:
os.listdir(ONNX_OUTPUT_PATH)


['config.json',
 'model.onnx',
 'tokenizer_config.json',
 'special_tokens_map.json',
 'vocab.txt',
 'tokenizer.json',
 'label_map.json']

In [ ]:
import torch
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT_PATH)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Your order has been shipped."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Predicted encoded label:", prediction)


Predicted encoded label: 3


In [ ]:
import torch
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT_PATH)
tokenizer = AutoTokenizer.from_pretrained(ONNX_OUTPUT_PATH)

text = "Congratulations! You won a free prize!"

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Predicted encoded label:", prediction)


Predicted encoded label: 0
